# 05. Aggregations, Grouping & Window Functions

In this notebook, you will learn how to summarize data, build pivot tables, and master **Window Functions** (one of the most asked topics in PySpark interviews and data engineering projects).

### Topics Covered:
1. **Simple Aggregations (`count`, `sum`, `avg`, `min`, `max`)**
2. **Grouping with `groupBy()` and multiple aggregations with `.agg()`**
3. **Pivot Tables with `.pivot()`**
4. **Window Functions Explained Step-by-Step**
   * Window specifications (`partitionBy`, `orderBy`)
   * Ranking: `row_number()`, `rank()`, `dense_rank()`
   * Finding Top N records per category
   * Analytics/Offsets: `lead()` and `lag()`
   * Running totals & cumulative sums
5. **Stopping the SparkSession to free memory**


---
## 1. Initializing SparkSession


In [ ]:


from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, sum, avg, min, max, count, countDistinct, round,
    row_number, rank, dense_rank, lead, lag
)
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType, DateType
)

spark = SparkSession.builder \
    .appName("05_Aggregations_and_Window_Functions") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("SparkSession ready!")


---
## 2. Creating Sample Dataset: Sales & Employee Performance


In [ ]:
sales_data = [
    ("Alice", "Engineering", "East", 85000, "2023-01-15"),
    ("Bob", "Marketing", "East", 62000, "2023-01-18"),
    ("Charlie", "Engineering", "West", 95000, "2023-02-01"),
    ("Diana", "Engineering", "West", 95000, "2023-02-10"), # Tied salary with Charlie
    ("Evan", "Marketing", "West", 71000, "2023-02-15"),
    ("Fiona", "Sales", "East", 55000, "2023-03-01"),
    ("George", "Sales", "East", 68000, "2023-03-05"),
    ("Hannah", "Sales", "West", 72000, "2023-03-12"),
    ("Ian", "Engineering", "East", 89000, "2023-03-20")
]

columns = ["name", "department", "region", "salary", "join_date"]

df = spark.createDataFrame(sales_data, columns)
df.show()


---
## 3. Simple Aggregations on the Entire DataFrame

You can calculate global metrics across the whole table without grouping.


In [ ]:
df.select(
    count("name").alias("total_employees"),
    countDistinct("department").alias("unique_depts"),
    round(avg("salary"), 2).alias("avg_salary"),
    min("salary").alias("min_salary"),
    max("salary").alias("max_salary"),
    sum("salary").alias("total_payroll")
).show()


---
## 4. Grouping Data with `groupBy()` and `.agg()`

* Always prefer `groupBy().agg(...)` over basic shortcut methods because it allows you to calculate multiple metrics simultaneously with custom aliases.


In [ ]:
# Group by Department: compute employee count, average salary, and max salary
dept_stats = df.groupBy("department").agg(
    count("name").alias("headcount"),
    round(avg("salary"), 2).alias("average_salary"),
    max("salary").alias("highest_salary"),
    min("salary").alias("lowest_salary")
)

dept_stats.show()


In [ ]:
# Grouping by multiple columns: Department AND Region
dept_region_stats = df.groupBy("department", "region").agg(
    count("name").alias("headcount"),
    sum("salary").alias("regional_payroll")
).orderBy("department", "region")

dept_region_stats.show()


---
## 5. Pivot Tables (`pivot`)

Pivoting rotates distinct values from a column into multiple new columns.
* Example: Compare department payroll broken down across East vs West regions.


In [ ]:
pivot_df = df.groupBy("department") \
    .pivot("region") \
    .agg(round(avg("salary"), 2))

print("Average salary by Department pivoted across Regions:")
pivot_df.show()


---
## 6. Window Functions: The Game Changer

### What is a Window Function?
Standard `groupBy()` collapses rows: 10 rows grouped by 3 departments become 3 rows.
A **Window Function** performs calculations across groups of rows (a "window"), but **preserves individual rows**!

### Anatomy of a Window:
1. `partitionBy("col")`: Divides data into groups (like `groupBy`).
2. `orderBy("col")`: Sorts rows within each group.
3. `rowsBetween(...)`: (Optional) Defines the sliding frame of rows to include.


---
## 7. Ranking Functions: `row_number()`, `rank()`, and `dense_rank()`

Let's understand the difference:
* **`row_number()`**: Always assigns a sequential number (1, 2, 3, 4...). Never has ties.
* **`rank()`**: Ties share the same rank, but the next rank **skips** numbers (1, 2, 2, 4).
* **`dense_rank()`**: Ties share the same rank, and the next rank **does not skip** numbers (1, 2, 2, 3).


In [ ]:
# Define window: Partition by department, ordered by salary descending
window_dept = Window.partitionBy("department").orderBy(col("salary").desc())

ranking_df = df.withColumn("row_num", row_number().over(window_dept)) \
               .withColumn("rank", rank().over(window_dept)) \
               .withColumn("dense_rank", dense_rank().over(window_dept))

print("Notice the difference in Engineering between Charlie and Diana (both earn 95,000):")
ranking_df.select("department", "name", "salary", "row_num", "rank", "dense_rank") \
          .orderBy("department", "salary", ascending=[True, False]) \
          .show()


---
## 8. Real-World Use Case: Top N Records Per Category

Find the **top 2 highest-paid employees in each department**.


In [ ]:
top_2_per_dept = ranking_df.filter(col("dense_rank") <= 2)
print("Top 2 highest paid employees per department:")
top_2_per_dept.select("department", "dense_rank", "name", "salary").show()


---
## 9. Analytic / Offset Functions: `lead()` and `lag()`

* **`lag(col, offset)`**: Accesses the value of a column from the **previous** row.
* **`lead(col, offset)`**: Accesses the value of a column from the **next** row.
* Perfect for calculating day-over-day changes, salary differences, or time elapsed.


In [ ]:
# Window ordered by salary across the whole company
window_salary = Window.orderBy("salary")

diff_df = df.withColumn("prev_lower_salary", lag("salary", 1).over(window_salary)) \
            .withColumn("next_higher_salary", lead("salary", 1).over(window_salary)) \
            .withColumn("gap_from_previous", col("salary") - col("prev_lower_salary"))

diff_df.select("name", "department", "salary", "prev_lower_salary", "gap_from_previous").show()


---
## 10. Cumulative Sums & Running Totals

Calculate running payroll total as employees join the company over time.


In [ ]:
# Window ordered by join_date, spanning from the beginning of time to the current row
running_window = Window.orderBy("join_date") \
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)

running_df = df.withColumn("running_total_payroll", sum("salary").over(running_window))

running_df.select("join_date", "name", "salary", "running_total_payroll").show()


---
## 11. Summary of Notebook 05
* Compute summary statistics using `df.select(agg_funcs)`.
* Group data using `df.groupBy("col").agg(...)`.
* Pivot categorical rows into columns with `.pivot()`.
* Use Window Functions to calculate rankings (`row_number`, `rank`, `dense_rank`), offsets (`lead`, `lag`), and running totals without collapsing rows!

---
## ⚠️ CRITICAL STEP: Stopping the SparkSession
Always release RAM and JVM background processes before moving to the next notebook.


In [ ]:
# Stop SparkSession to free memory
spark.stop()
print("SparkSession stopped successfully! RAM released.")
